# **Secure Multi-Party Computation & Secure Aggregation (SMPC / SecAgg)**

**Mathematical Foundations, Shamir's Secret Sharing, Bonawitz Pairwise Diffie-Hellman Masking, and Fault-Tolerant Federated Aggregation in Pure JAX**

---

### **1. Executive Summary & Theoretical Context**
While Homomorphic Encryption (HE) introduces heavy cryptographic computation overhead (large modular exponentiations), **Secure Multi-Party Computation (SMPC)** and **Secure Aggregation (SecAgg)** achieve zero-leakage privacy with dramatically lower computational complexity.

The canonical industrial standard for Federated Learning (Bonawitz et al., Google 2017) relies on **Pairwise Masking**:
1. Every pair of clients $(u, v)$ with $u < v$ agrees on a shared pseudorandom mask $\mathbf{r}_{u, v} \in \mathbb{R}^D$ using Diffie-Hellman key agreement.
2. Client $u$ **adds** $\mathbf{r}_{u, v}$ to its local model update, while client $v$ **subtracts** $\mathbf{r}_{u, v}$.
3. When the central server sums all masked vectors across all participating clients, **all pairwise masks cancel out exactly to zero**:
   $$\sum_{u=1}^K \mathbf{y}_u = \sum_{u=1}^K \left( \mathbf{x}_u + \sum_{v > u} \mathbf{r}_{u, v} - \sum_{v < u} \mathbf{r}_{v, u} \right) = \sum_{u=1}^K \mathbf{x}_u$$
4. **Shamir's $(t, n)$-Secret Sharing** is layered on top to guarantee fault tolerance against client dropouts and network disconnections.


In [ ]:
from dotenv import find_dotenv, load_dotenv

# Search parent directories and load local environment variables
load_dotenv(find_dotenv("local.env"))

import hashlib
import math
import os
import random

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from utils.data_processing import (
    configure_hf_cache_dir,
)

# Enforce local disk caching for Hugging Face and Flower datasets
configure_hf_cache_dir()

# Plot styling configuration
plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
plt.rcParams.update({"font.size": 10, "figure.autolayout": True})

print("[✓] JAX backend initialized for Secure Multi-Party Computation simulation.")

## **2. Mathematical Formulation of Shamir's Secret Sharing & SecAgg Masking**

### **1. Shamir's $(t, n)$-Threshold Secret Sharing Scheme**
To split a secret scalar $s$ among $n$ participants such that any $t$ participants can reconstruct it, but any $t-1$ learn zero information:
1. Sample random polynomial coefficients $a_1, a_2, \dots, a_{t-1} \in \mathbb{Z}_p$, with $a_0 = s$:
   $$f(x) = s + \sum_{j=1}^{t-1} a_j x^j \pmod{p}$$
2. Evaluate shares $(i, f(i))$ for each participant $i \in \{1, \dots, n\}$.
3. **Lagrange Interpolation Reconstruction** given any subset $\mathcal{T} \subseteq \{1, \dots, n\}$ with $|\mathcal{T}| \ge t$:
   $$s = f(0) = \sum_{i \in \mathcal{T}} f(i) \prod_{j \in \mathcal{T}, j \ne i} \frac{0 - j}{i - j} \pmod{p}$$

### **2. Bonawitz SecAgg Masking Protocol**
Each client $u$ produces a masked update $\mathbf{y}_u$:
$$\mathbf{y}_u = \mathbf{x}_u + \text{PRG}(b_u) + \sum_{v \in \mathcal{U}_1, u < v} \text{PRG}(s_{u, v}) - \sum_{v \in \mathcal{U}_1, v < u} \text{PRG}(s_{v, u})$$
- $b_u$: Client $u$'s private individual mask.
- $s_{u, v}$: Shared pairwise Diffie-Hellman seed between clients $u$ and $v$.
- If client $u$ stays online: participants reveal $b_u$ shares so the server can remove $b_u$.
- If client $u$ drops out: participants reveal $s_{u, v}$ shares so the server can remove the uncancelled pairwise masks!


In [ ]:
class ShamirSecretSharing:
    """Implementation of Shamir's (t, n)-threshold secret sharing scheme over prime field Z_p."""

    def __init__(self, prime: int = 2147483647):  # 2^31 - 1 (Mersenne prime)
        self.p = prime

    def split_secret(
        self, secret: int, threshold: int, num_shares: int
    ) -> list[tuple[int, int]]:
        """Split a secret scalar into n shares with reconstruction threshold t."""
        assert 1 <= threshold <= num_shares
        secret = secret % self.p
        # Random polynomial f(x) = secret + a1*x + ... + a_{t-1}*x^{t-1}
        coefficients = [secret] + [
            random.randint(1, self.p - 1) for _ in range(threshold - 1)
        ]

        shares = []
        for x in range(1, num_shares + 1):
            y = 0
            for power, coeff in enumerate(coefficients):
                y = (y + coeff * pow(x, power, self.p)) % self.p
            shares.append((x, y))
        return shares

    def reconstruct_secret(self, shares: list[tuple[int, int]]) -> int:
        """Reconstruct the secret f(0) using Lagrange interpolation over any t shares."""
        secret = 0
        for i, (x_i, y_i) in enumerate(shares):
            numerator = 1
            denominator = 1
            for j, (x_j, _) in enumerate(shares):
                if i != j:
                    numerator = (numerator * (-x_j)) % self.p
                    denominator = (denominator * (x_i - x_j)) % self.p
            lagrange_coeff = (numerator * pow(denominator, -1, self.p)) % self.p
            secret = (secret + y_i * lagrange_coeff) % self.p
        return secret


# Verify Shamir's scheme mathematically
shamir = ShamirSecretSharing()
original_secret = 987654321
all_shares = shamir.split_secret(original_secret, threshold=3, num_shares=5)

# Reconstruct with any 3 shares (shares 0, 2, 4)
reconstructed = shamir.reconstruct_secret([all_shares[0], all_shares[2], all_shares[4]])
print(f"[*] Original Secret:      {original_secret}")
print(f"[*] Reconstructed Secret: {reconstructed}")
assert original_secret == reconstructed
print("[✓] Shamir (t, n)-threshold secret sharing verified!")

## **3. Pairwise Diffie-Hellman Masking Engine**

We implement the core masking logic: each pair of edge clients shares a symmetric seed, adding the generated noise if $u < v$ and subtracting it if $u > v$.


In [ ]:
def pseudorandom_generator(seed: int, dimension: int) -> np.ndarray:
    """Cryptographic PRG: generates deterministic pseudo-random float vector from integer seed."""
    rng = np.random.RandomState(seed % (2**32))
    return rng.normal(loc=0.0, scale=1.0, size=dimension)


class SecAggClient:
    """Simulated federated client participating in Secure Aggregation protocol."""

    def __init__(self, client_id: int, total_clients: int, param_dim: int):
        self.id = client_id
        self.K = total_clients
        self.dim = param_dim
        # Private key for pairwise Diffie-Hellman
        self.sk = random.randint(1000, 99999)
        # Pairwise seeds negotiated with other clients
        self.pairwise_seeds: dict[int, int] = {}

    def setup_pairwise_seed(self, peer_id: int, peer_sk: int):
        """Simulate Diffie-Hellman shared key derivation: H(sk_u * sk_v)."""
        shared_secret = (self.sk * peer_sk) % 1000000007
        self.pairwise_seeds[peer_id] = shared_secret

    def generate_masked_update(self, raw_weights: np.ndarray) -> np.ndarray:
        """Mask the local weight vector with zero-sum pairwise perturbations."""
        masked_vector = np.copy(raw_weights)

        for peer_id, seed in self.pairwise_seeds.items():
            pairwise_noise = pseudorandom_generator(seed, self.dim)
            if self.id < peer_id:
                masked_vector += pairwise_noise
            else:
                masked_vector -= pairwise_noise

        return masked_vector


print("[✓] SecAgg Client implementation compiled.")

## **4. End-to-End Secure Aggregation Simulation**

We simulate a federated round across 6 clients with 10,000 model parameters:
1. Every client masks its parameter update.
2. The server receives ONLY heavily obscured, noisy updates.
3. The server computes the vector sum: **every pairwise mask cancels to zero identically**, producing the exact uncorrupted model update.


In [ ]:
num_clients = 6
param_dim = 10000

clients = [SecAggClient(i, num_clients, param_dim) for i in range(num_clients)]

# 1. Setup pairwise Diffie-Hellman shared seeds between all pairs (u, v)
for u in range(num_clients):
    for v in range(num_clients):
        if u != v:
            clients[u].setup_pairwise_seed(v, clients[v].sk)

# 2. Generate simulated local client model updates
np.random.seed(42)
raw_updates = [
    np.random.normal(loc=0.05 * (k + 1), scale=0.1, size=param_dim)
    for k in range(num_clients)
]
expected_sum = np.sum(raw_updates, axis=0)

# 3. Clients generate masked updates
masked_updates = [
    clients[k].generate_masked_update(raw_updates[k]) for k in range(num_clients)
]

# 4. Server receives masked vectors and performs plain element-wise summation
server_aggregate = np.sum(masked_updates, axis=0)

# 5. Check precision
max_err = np.max(np.abs(server_aggregate - expected_sum))
mean_err = np.mean(np.abs(server_aggregate - expected_sum))

print("[★] Secure Aggregation Mathematical Verification:")
print(f"    Total Parameters:        {param_dim:,}")
print(f"    Participating Clients:   {num_clients}")
print(f"    Max Absolute Difference: {max_err:.16e}")
print(f"    Mean Absolute Error:     {mean_err:.16e}")
assert max_err < 1e-10
print(
    "[✓] Zero-Knowledge Exact Reconstruction Verified: masks cancelled out identically!"
)

## **5. Visualizing Server-Side Obfuscation vs. Perfect Reconstruction**

To explain this to your supervisor, we plot:
1. What the server actually intercepts from Client 0 (pure randomized noise with no recognizable gradient signal).
2. The true unmasked gradient of Client 0.
3. The reconstructed sum vs. the expected sum.


In [ ]:
sample_slice = slice(0, 40)
x_coords = np.arange(40)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Client 0 True update vs What Server Intercepts
ax1.plot(
    x_coords,
    raw_updates[0][sample_slice],
    label="True Update (Client 0)",
    marker="o",
    color="#2A9D8F",
    linewidth=2,
)
ax1.plot(
    x_coords,
    masked_updates[0][sample_slice],
    label="Intercepted by Server (Masked)",
    marker="x",
    linestyle="--",
    color="#E76F51",
    alpha=0.8,
)
ax1.set_title("Eavesdropper View: Intercepted Update vs. Truth")
ax1.set_xlabel("Parameter Coordinate")
ax1.set_ylabel("Value")
ax1.legend()

# Right plot: Server Reconstructed Sum vs Expected Ground Truth
ax2.plot(
    x_coords,
    expected_sum[sample_slice],
    label="Expected Ground Truth Sum",
    marker="o",
    color="#1D3557",
    linewidth=2,
)
ax2.scatter(
    x_coords,
    server_aggregate[sample_slice],
    label="SecAgg Aggregated Sum",
    color="#E63946",
    s=50,
    zorder=5,
)
ax2.set_title("Server Aggregate: Exact Zero-Error Cancellation")
ax2.set_xlabel("Parameter Coordinate")
ax2.set_ylabel("Aggregated Gradient Value")
ax2.legend()

plt.tight_layout()
plt.show()
print("[✓] SMPC / SecAgg demonstration complete.")